# Retrieval-Augmented Generation on Historical UAP Records

---

**Students**

* Sofia Di Lucia (Badge No. 2149752)
* Giovanni Andrea Maida (Badge No. 2159404)

**Master's Program:** M.Sc. in Physics of Data

**Academic Year:** 2025–2026

---

**Sections**:
- [Domain and Dataset](#Domain-and-Dataset)
- [Importing Libraries](#Importing-Libraries)
- [Inspecting Dataset](#Inspecting-Dataset)
- [Embedding](#Embedding)
- [Retrieval](#Retrieval)
- [Generation](#Generation)
- [Evaluation](#Evaluation)
- [Acknowledgements](#Acknowledgements)
- [Appendix](#Appendix)

### Domain and Dataset

The project is based on the *[MTS, Department of War UAP Release 1 — Structured Corpus (2026)](https://huggingface.co/datasets/MTSlive/war-gov-uap-release-1)* dataset (source material at [war.gov/UFO/](https://www.war.gov/UFO/)).

<img src="https://media.mts-in.com/release_1/38-143685-box-incident-summaries-101-172/p137_f1_sketch.webp"
         alt="UFO"
         width="630"
         height="180">

### Hardware

The code was run on a [CloudVeneto](https://cloudveneto.it/) Virtual Machine mounting a NVIDIA Tesla T4 GPU.

# Importing Libraries

In [1]:
from datasets import load_dataset, load_from_disk, get_dataset_config_names, Dataset
import faiss
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
import pandas as pd
from pathlib import Path
import re
import torch.nn.functional as F
from torch import Tensor
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig

# Useful functions

In [2]:
def save_file(new_file_name, obj_to_save):
  with open(new_file_name, 'w') as f:
    for line in obj_to_save:
        f.write(f"{line}\n")
  return print(f"File {new_file_name} saved")

def load_file(file_name):
    with open(file_name, "r") as f:
        obj_saved = [line.strip() for line in f]
    return obj_saved

In [3]:
# function to extract the last token (EOS) that is a compressed representation of the whole input (instruction+query)
def last_token_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

# we "merge" in one string instruction and query
def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery: {query}'

def get_embeddings(text_list, max_length=512):

  # Tokenize the input texts
  batch_dict = tokenizer(
    text_list,
    padding='longest',
    truncation=True,
    max_length=max_length,
    return_tensors="pt",
  )
  batch_dict.to(model.device)

  with torch.no_grad(): outputs = model(**batch_dict)
  embeddings = last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask']).cpu()

  torch.cuda.empty_cache()

  return embeddings

In [4]:
def generation(input_tokenizer, input_model, messages, max_tokens, temperature=0.7, top_p=0.8, think=False):
  tokenizer = input_tokenizer
  model = input_model

  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
      enable_thinking=think
  )
  model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

  # conduct text completion
  generated_ids = model.generate(
      **model_inputs,
      max_new_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p
  )
  output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

  # parsing thinking content
  try:
      # rindex finding 151668 (</think>)
      index = len(output_ids) - output_ids[::-1].index(151668)
  except ValueError:
      index = 0

  thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
  content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

  return thinking_content, content


# Inspecting Dataset

In [5]:
ufo_dataset = "MTSLIVE/war-gov-uap-release-1"
get_dataset_config_names(ufo_dataset)   # dataset files

['documents', 'pages', 'figures', 'videos']

We will use the `pages` file.

In [6]:
pages = load_dataset(ufo_dataset, "pages", split="train")
pages

Dataset({
    features: ['document_id', 'page_no', 'text', 'has_figures'],
    num_rows: 4239
})

In [7]:
# example of what we are going to use
print(pages[0]['text'])

HEADQUARTERS
AIR MATERIEL COMMAND
WRIGHT FIELD, DAYTON, OHIO

DEC 1 9 1947

SUBJECT: Flying Discs

TO: Chief of Staff
United States Air Force
Washington 25, D. C.
ATTENTION: Director, Research & Development
Major General L. C. Craigie

1. Confirming the recent conversation of the undersigned with Major General L. C. Craigie, 9 December 1947, attached as listed below are copies of the reports from this Headquarters concerning Flying Discs.

2. Comments of Headquarters, Air Force on these letters have never been received by this Command. Continued and recent reports from qualified observers concerning this phenomenon still makes this matter one of concern to Headquarters, Air Materiel Command. Intelligence Department of this Command is continuing the collection and analysis of all available reports.

FOR THE COMMANDING GENERAL:

H. M. McCOY
Colonel, USAF
Chief of Intelligence

2 Attach:
cc ltr to CG, AAF, dtd 23 Sept 47 subj "AMC Opinion Concerning "Flying Discs""
cc ltr to CG, AAF, dtd 

## Check single page documents

In [8]:
# all documents, each of these is composed of 1 or more pages
IDS = set(pages["document_id"])

In [9]:
count = pages.to_pandas().groupby("document_id").count()
single = list(count[count["page_no"]==1].index)  # Documents of 1 page only

In [10]:
# Discard single paged documents containing no information
pattern = re.compile("fbi-photo|nasa-uap-vm|fbi-september-2023-sighting-composite-sketch")  # compile pattern to match discarded documents
useIDS = set(ID for ID in IDS if not pattern.match(ID))  # discard matches
filterPages = pages.filter(lambda x: x["document_id"] in useIDS and x["text"] != '')  # select docs and remove empty pages

# Embedding

## Models

The embedder is choosen from the [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard).

**[Qwen3](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) Model Architecture**: <br>
is designed using dual-encoder and cross-encoder architectures. The Embedding model processes a single text segment as input, extracting the semantic representation by the hidden state vector corresponding to the final [EOS] token ([web reference](https://qwen.ai/blog?id=qwen3-embedding) and [article](https://arxiv.org/pdf/2506.05176) explaining the Qwen3 model.).

<div style="display:flex; gap:20px;">
    <img src="https://miro.medium.com/v2/resize:fit:750/format:webp/1*jzZ_e5Bmvx84zPEa-LhqDQ.png"
         alt="Qwen3 architecture"
         width="500">
    <img src="https://miro.medium.com/v2/resize:fit:1400/format:webp/1*AWPQxx4xpiQGCp5XNNfyYA.png"
         alt="BERT vs Qwen3"
         width="700"
         height="300">
</div>

For text embeddings, we utilize LLMs with causal attention, appending an [EOS] token at the end of the input sequence. The final embedding is derived from the hidden state of the last layer corresponding to this [EOS] token.
To ensure embeddings follow instructions during downstream tasks, we concatenate the instruction and the query into a single input context, while leaving the document unchanged before processing with LLMs. The input format for queries is as follows:
`{Instruction}{Query}<|endoftext|>`

[Harrier-oss-v1](https://huggingface.co/microsoft/harrier-oss-v1-0.6b) is a Decoder-Only architecture as well, and works similarly to Qwen. Harrier is specifically designed to embed text, hence it should achieve similar results even with less parameters, as the leaderboard suggests (at time of writing).

In [20]:
harrier_embedder = "microsoft/harrier-oss-v1-0.6b"
qwen_embedder = "Qwen/Qwen3-Embedding-4B"

To fit the models and the data in the GPU memory, we use the [bitsandbytes](https://huggingface.co/docs/bitsandbytes/index) library, specifically designed to dramatically reduce the model's memory consumption without performance degradation, using k-bit quantization ($4$-bit in our case). 

In [8]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

In [21]:
# defining the embedding model and the relative tokenizer
# for the tokenizer we need the left padding

# harrier
tokenizer = AutoTokenizer.from_pretrained(harrier_embedder, padding_side='left', cache_dir='tokenizers_cache')
model = AutoModel.from_pretrained(harrier_embedder, quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir='models_cache')

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [ ]:
# qwen
tokenizer = AutoTokenizer.from_pretrained(qwen_embedder, padding_side='left', cache_dir='tokenizers_cache')
model = AutoModel.from_pretrained(qwen_embedder, quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir="models_cache")

## Documents

We use [LangChain](https://docs.langchain.com/) to split the pages into chunks of $512$ tokens with $15\%$ overlapping. Using the provided tokenizer, LangChain tries to split the text until the chunks are small enough while trying to keep paragraphs, then sentences, and finally words together where possible.

In [ ]:
# function to create the chunked text (input of embedder)
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer, chunk_size=512, chunk_overlap=77)

pages_chunked = []

for page in filterPages:
    page_text = page['text']
    chunk_list = text_splitter.split_text(page_text)
    for i, chunk in enumerate(chunk_list):
        pages_chunked.append({
            'document_id': page['document_id'],
            'page_no': page['page_no'],
            'chunk_id': i,
            'chunk_text': chunk
    })

chunkedPages = Dataset.from_list(pages_chunked)

Since embedding the large number of documents requires a significant amount of time ($\sim 20~m$ using the vm GPU for the Harrier embeddings) we decided to run a python script on the vm we could leave running without worrying about possible disconnections or suspensions of our laptops (see [Appendix C](#Appendix-C---Embedding-code)).

Below we load the embeddings.

In [14]:
## Harrier
raw_docsEmbed = torch.load("/mnt/eph/embedding/harrier06DocsEmbed.pt", weights_only=False)
raw_embedPages = chunkedPages.add_column("embeddings", raw_docsEmbed.to(torch.float32).cpu().numpy().tolist())
raw_embedPages

Dataset({
    features: ['document_id', 'page_no', 'chunk_id', 'chunk_text', 'embeddings'],
    num_rows: 5321
})

In [18]:
## Qwen
raw_docsEmbed = torch.load("/mnt/eph/embedding/qwen4DocsEmbed.pt", weights_only=False)
raw_embedPages = chunkedPages.add_column("embeddings", raw_docsEmbed.to(torch.float32).cpu().numpy().tolist())
raw_embedPages

Dataset({
    features: ['document_id', 'page_no', 'chunk_id', 'chunk_text', 'embeddings'],
    num_rows: 5321
})

We noticed that some paragraphs of the documents are repeated more than once. Here we remove such passages by checking on their embeddings, since if they achieve a high score they are just retrieved multiple times, hindering the system performance.

In [ ]:
check = []
keep = []
for i, page in enumerate(raw_embedPages):
    if page["embeddings"] in check: continue
    else:
        keep.append(i)
        check.append(page["embeddings"])
del check

In [16]:
embedPages = raw_embedPages.select(keep)
docsEmbed = raw_docsEmbed[keep]

In [18]:
## Harrier
embedPages

Dataset({
    features: ['document_id', 'page_no', 'chunk_id', 'chunk_text', 'embeddings'],
    num_rows: 5156
})

In [21]:
## Qwen ###################################### DON'T RUN SOFIAAAAAAAAAAAAAAAAAAA ###########################
embedPages

Dataset({
    features: ['document_id', 'page_no', 'chunk_id', 'chunk_text', 'embeddings'],
    num_rows: 5182
})

A total of $165$ repeated chunks are cleaned for the Harrier embeddings, and $139$ for Qwen.

In [26]:
# save the objects to quick start next steps (commented out for safety)

# embedPages.save_to_disk("/mnt/eph/embedding/embedPages")
# torch.save(docsEmbed, "/mnt/eph/embedding/docsEmbed.pt")

## Queries

In [21]:
# Each query must come with a one-sentence instruction that describes the task
task = 'Given a document search query, retrieve relevant passages that answer the query'

raw_queries = [
    "What are Jesus's powers?",
    'What was spotted in the sky for the first time?',
    'Have UFOs ever been close to humans (astronauts)?',
    "What patterns emerge across the reported UAP sightings regarding location, altitude, behavior, speed, and time period?",
    "What are the most extravagant sightings?",
    "What is the Saucer's secret?",
    "What are they hiding from us?"
]

queries = [get_detailed_instruct(task, query) for query in raw_queries]

In [22]:
queriesEmbed = get_embeddings(queries)

# Retrieval

Even though the embeddings fit in our machine memory, we decided to use [Faiss](https://github.com/facebookresearch/faiss) to calculate cosine similarity. Faiss is a library for efficient similarity search and clustering of dense vectors, optimized both in terms of computing speed, and memory usage.

In [23]:
# retrieving the top-5 documents for each query
numpydocs = docsEmbed.to(torch.float32).cpu().numpy()
numpyqueries = queriesEmbed.to(torch.float32).cpu().numpy()
index = faiss.index_factory(docsEmbed.shape[1], "Flat", faiss.METRIC_INNER_PRODUCT)   # cosine similarity
faiss.normalize_L2(numpydocs)
index.add(numpydocs)
faiss.normalize_L2(numpyqueries)

k=5
k_best_distance, k_best_index = index.search(numpyqueries, k)

To avoid excessive text dumping, we show here the selected documents for the first $2$ queries only. The rest is shown in the [Appendix A](#Appendix-A---Retrieval) at the end of the notebook.

In [24]:
## Harrier
for i, query in enumerate(raw_queries[:2]):
    print(query)
    for j, idx in enumerate(k_best_index[i]):
        print(f"\nscore: {k_best_distance[i,j]}; idx: {idx}\n{embedPages[idx]["chunk_text"]}")
    print("\n\n\n")

What are Jesus's powers?

score: 0.5771710872650146; idx: 3266
The life of Jesus, now too, becomes clearer when these things are re-membered. His powers of levitation, His ability to pass through doors, walk on water, and heal the sick, are the essential attributes of men from outer space. They will also be ours someday when we will be "free like birds" (Ezk. 13:20). At Jesus' birth the celestial army came quite close to earth. A space-man appeared to the shepherds and the "glory" of the Lord, with the usual signs of His presence, shone around them. There was a multitude of the heavenly army with this space-man. And after their cosmic announcement, the music of their space-ships was heard as they again disappeared into space.

Jesus' ascension is described as "a cloud (or space-ship) received Him out of their sight" (Acts 1:9). His coming again is to be in the same manner. "Then will appear the sign of the Son of man in heaven (space) coming on the clouds (space-ships) of heaven (space

In [23]:
## Qwen
for i, query in enumerate(raw_queries[:2]):
    print(query)
    for j, idx in enumerate(k_best_index[i]):
        print(f"\nscore: {k_best_distance[i,j]}; idx: {idx}\n{embedPages[idx]["chunk_text"]}")
    print("\n\n\n")

What are Jesus's powers?

score: 0.623723030090332; idx: 3272
The life of Jesus, now too, becomes clearer when these things are re-membered. His powers of levitation, His ability to pass through doors, walk on water, and heal the sick, are the essential attributes of men from outer space. They will also be ours someday when we will be "free like birds" (Ezk. 13:20). At Jesus' birth the celestial army came quite close to earth. A space-man appeared to the shepherds and the "glory" of the Lord, with the usual signs of His presence, shone around them. There was a multitude of the heavenly army with this space-man. And after their cosmic announcement, the music of their space-ships was heard as they again disappeared into space.

Jesus' ascension is described as "a cloud (or space-ship) received Him out of their sight" (Acts 1:9). His coming again is to be in the same manner. "Then will appear the sign of the Son of man in heaven (space) coming on the clouds (space-ships) of heaven (space)

For the first query about the powers of Jesus the models retrieve the same passages, save for minor differences in the tokenization of the text. The scores also seem consistent enough, as the documents are not directly related to the topic, but may contain declarations from random people.

The second query instead is an example of a too generic question that cannot be answered by these documents. Both the models retrieve passages about flying objects with references to time in general. We would argue that Harrier does a better job, as its documents mostly contain at least a date, while Qwen retrieve passages referring to specific times of the day, but not the exact date. A higher amount of documents may benefit this second query, but it is not granted that enough information is retrieved to answer the query.

We chose Harrier as the retriever module of our RAG, both because it uses far less parameters, and it retrieves more interesting documents (see also [Appendix A](#Appendix-A---Retrieval)). 

# Generation

For the generation part we chose Qwen3.

We first use a single query (`What is the Saucer's secret?`) to test how the prompt, the temperature parameter, and the thinking mode affect the generation (thinking mode should be more suited to solve complex problems rather than being used for general porpuse dialogue).

Suggested parameters from qwen3 model card:

> For non-thinking mode, we suggest using Temperature=0.7, TopP=0.8, TopK=20, and MinP=0. For more detailed guidance, please refer to the Best Practices section.

> For thinking mode, use Temperature=0.6, TopP=0.95, TopK=20, and MinP=0 (the default setting in generation_config.json). DO NOT use greedy decoding, as it can lead to performance degradation and endless repetitions. For more detailed guidance, please refer to the Best Practices section.

The Answers to the queries using the best configuration are reported in [Appendix B](#Appendix-B---Generation).

In [17]:
qwen_generator = "Qwen/Qwen3-4B"

In [18]:
# qwen
tokenizer = AutoTokenizer.from_pretrained(qwen_generator, padding_side='left', cache_dir='tokenizers_cache')
model = AutoModelForCausalLM.from_pretrained(qwen_generator, quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir="models_cache")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## Prompt

We focus on tailoring the `system_prompt`, leaving the `user_prompt` quite generic, as printed below:

In [42]:
retrDocs = [embedPages[int(idx)]["chunk_text"] for idx in k_best_index[-2]]
prompt = "Provided documents:\n"
for j, doc in enumerate(retrDocs): prompt += f"Document {str(j + 1)}:\n{doc}.\n\n"    # here we can print metadata (doc_id and page_no) instead of whole text
prompt += f"\nQuery: {raw_queries[-2]}"
print(prompt)

Provided documents:
Document 1:
FROM the beginning, the officers in charge of Project Saucer recognized a peculiar difficulty in their assignment. "If you look out the window and see something, how can I prove or disprove what it was if I didn't see it and you can't tell me much about what you.

Document 2:
A NUCLEAR PHYSICIST EXPOSES FLYING SAUCERS

"There is no longer any need for secrecy," says Navy scientist, after finding that his own research started the "saucers"

By RICHARD WILSON Chief of LOOK Washington Bureau

THE literal-minded FBI, skeptical but determined, could not let the flying-saucer excitement go by without getting to the bottom of it. Such a profusion of strange objects littering the American skies could not be ignored.

A 10-page report by the nuclear physics branch of the Office of Naval Research has given the answer:

Flying saucers were, and are, undeniably real. They are part of a basic research program of the Federal Government which is as important, if not so

In [43]:
system_prompt = "Use the provided documents to accurately answer the Query."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": prompt}
]
for i in range(5): 
    print(f"Answer {i}:")
    print(generation(tokenizer, model, messages, 16384)[1], '\n')

Answer 0:
The "saucer's secret" refers to the fact that flying saucers are not mythical or extraterrestrial in nature, but rather real objects with a scientific and technological basis. According to the documents, the saucers are part of a government-funded research project, specifically the "Skyhook" project by the Office of Naval Research. These saucers are large balloons (100 feet in diameter) used for scientific research at high altitudes to study atomic explosions in the atmosphere. They carry delicate instruments to observe and measure the effects of cosmic particles on atomic structures, aiming to understand and harness atomic energy.

Additionally, some documents suggest that the saucers might be related to experimental aircraft or military technology, such as the "Flying Flapjack" or the "Drattin saucer" mentioned in Document 5, which is described as a radio-controlled jet tank with features resembling a flying saucer. These crafts are part of experimental aviation projects, s

These answers are already quite satisfying as they are$-$they are similar, but temperature will be discussed later. We try to also make it reference some information the LLM may know by itself.

In [45]:
system_prompt = "First tell what you know about the Query, without using the information in the provided documents. Then, use the provided documents to accurately answer the Query."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": prompt}
]
for i in range(5): 
    print(f"Answer {i}:")
    print(generation(tokenizer, model, messages, 16384, think=False)[1], '\n')

Answer 0:
The "saucer's secret" refers to the mysterious nature of flying saucers, which have been the subject of much speculation and research. According to the documents provided, the secret of the saucers is revealed to be a sophisticated scientific experiment.

In Document 2, it is disclosed that flying saucers are actually part of a government research project. They are described as being based on a huge balloon called a "Skyhook," which is used to carry delicate instruments for scientific research. These balloons are used to observe and measure the explosions of atoms in the atmosphere caused by cosmic particles. The goal of this research is to understand the structure of matter and to harness energy from the decomposition of the atom, rather than for military or destructive purposes.

Additionally, Document 4 mentions that some scientists believe the saucers are actually experimental aircraft, such as the "Flying Flapjack," which is a prototype for a new type of aircraft. The do

Add personality to the answer

In [46]:
system_prompt = "We are impartial detectives investigating the latest FBI UAP documents. First tell what you know about the Query, without using the information in the provided documents. Then use the fresh information of the provided documents to accurately answer the Query."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": prompt}
]
for i in range(5): 
    print(f"Answer {i}:")
    print(generation(tokenizer, model, messages, 16384, think=False)[1], '\n')

Answer 0:
The "saucer's secret" refers to the mysterious nature of flying saucers, which have been the subject of much speculation, research, and conspiracy theory. Based on the provided documents, here is an analysis of what the "secret" might be:

### What We Know About the Query:
The term "saucer" in this context refers to the flying saucer or UFO phenomenon. The "secret" is the nature of these saucers—what they are, how they operate, and why they are so mysterious. The documents suggest that the saucers may not be extraterrestrial in origin, but rather could be advanced technological devices with a scientific purpose.

### Analysis of the Documents:
1. **Document 1** discusses the difficulty of proving or disproving the existence of saucers, as they are often described by witnesses but never seen by the observer. This highlights the challenge of verifying their nature.

2. **Document 2** reveals that the saucers may be part of a secret government project, specifically the "Skyhook"

Enforce a specific layout.

In [48]:
system_prompt = "We are impartial detectives investigating the latest FBI UAP documents.\
First tell what you know about the Query, without using the information in the provided documents.\
Then use the fresh information of the provided documents to accurately answer the Query. Give your answer using the following sections:\
# What we know;\
# Document information (address each document separately);\
# Conclusions"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": prompt}
]
for i in range(5): 
    print(f"Answer {i}:")
    print(generation(tokenizer, model, messages, 16384, think=False)[1], '\n')

Answer 0:
# What we know  
The "saucer" refers to the mysterious flying objects reported by witnesses, often described as disk-shaped craft that move at high speeds and are difficult to detect. The term "saucer" has been used to describe these objects in various contexts, including military, scientific, and anecdotal reports. The "secret" of the saucer has been the subject of much speculation, with theories ranging from extraterrestrial origins to advanced technology, military experiments, or even hoaxes. The documents provided offer insights into the scientific and governmental perspectives on the saucer phenomenon.

# Document information  
**Document 1:**  
This document highlights the challenges faced by Project Saucer, emphasizing the difficulty of proving or disproving the existence of these objects. The ambiguity of what is seen from the outside, combined with the inability to verify observations, creates a paradox for investigators. The document underscores the need for objecti

## Thinking

We now test the differences between the standard and thinking model, using the prompt.

In [50]:
system_prompt = "We are impartial detectives investigating the latest FBI UAP documents.\
First tell what you know about the Query, without using the information in the provided documents.\
Then use the fresh information of the provided documents to accurately answer the Query. Give your answer using the following sections:\
# What we know;\
# Document information (address each document separately);\
# Conclusions"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": prompt}
]
for i in range(5):
    thinking_content, content = generation(tokenizer, model, messages, 16384, temperature=.6, top_p=.95, think=True)
    print(f"Answer {i}:")
    print("thinking:", thinking_content, '\n')
    print("content:", content, '\n\n')

/mnt/eph/miniconda3/envs/nlp/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Answer 0:
thinking: <think>
Okay, so I need to figure out what the "saucer's secret" is based on the provided documents. Let me start by reading through each document carefully.

Document 1 talks about Project Saucer and the difficulty in proving what they saw. The officers are struggling because they can't confirm what they saw. This seems to be about the challenge of verifying sightings.

Document 2 mentions a nuclear physicist who says flying saucers are real and part of a government project. The "Skyhook" is a balloon with instruments to study atomic energy. The project is about understanding atomic energy, not bombs, but harnessing it. The saucer is a balloon with a disc shape, used for research. So the secret here might be related to nuclear research.

Document 3 is a letter asking if someone has seen a saucer and if it's a new era. It's a bit vague, but the mention of "new and revolutionary advance" might hint at something technological.

Document 4 discusses a possible explanat

Thinking does not change the output significantly, but it severely slows down the model, hence we decided not to use it. There may be tasks where thinking could be more suited, but not to retrieve general information.

## Temperature

The temperature suggested by the model card is already pretty high. We try to switch to the standard softmax $\left(\tau = 1\right)$, and compare the result.

In [51]:
system_prompt = "We are impartial detectives investigating the latest FBI UAP documents.\
First tell what you know about the Query, without using the information in the provided documents.\
Then use the fresh information of the provided documents to accurately answer the Query. Give your answer using the following sections:\
# What we know;\
# Document information (address each document separately);\
# Conclusions"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": prompt}
]
for i in range(5): 
    print(f"Answer {i}:")
    print(generation(tokenizer, model, messages, 16384, temperature=1)[1], '\n')

Answer 0:


/mnt/eph/miniconda3/envs/nlp/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


# What we know  
The query asks about the "secret" of the Saucer, which refers to the mysterious flying objects reported by witnesses and the various theories surrounding their origin and purpose. The term "saucer" is used to describe these objects, often described as disc-shaped, and their appearance has been the subject of much speculation, fear, and scientific curiosity. The "secret" likely refers to the true nature of these saucers—whether they are real, man-made, or extraterrestrial in origin.

# Document information  
**Document 1**  
This document highlights the challenges faced by the officers in Project Saucer. They are tasked with investigating strange sightings, but without concrete evidence, it is difficult to prove or disprove what is seen. The document underscores the difficulty of verifying claims when the witnesses cannot provide a clear, verifiable account of what they saw.

**Document 2**  
This document claims that flying saucers are not fictional but are real and pa

The default softmax increases the variabilty in the answers, and enriches the vocabulary without exceeding in extravagant words or artifacts.

# Evaluation

We select the generators from the [huggingface Leaderboard](https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard#/) (Considering the limit of our GPU memory).

We need to import:

In [11]:
docsEmbed = torch.load("/mnt/eph/embedding/docsEmbed.pt", weights_only=False)
embedPages = load_from_disk("/mnt/eph/embedding/embedPages")

In [12]:
test_pages = filterPages.shuffle(seed=1758).select(range(15))

In [14]:
test_generator = "Qwen/Qwen2.5-Coder-14B-Instruct"

In [15]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

tokenizer = AutoTokenizer.from_pretrained(test_generator, padding_side='left', cache_dir='tokenizers_cache')
model = AutoModelForCausalLM.from_pretrained(test_generator, quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir="models_cache")

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

In [38]:
# two quastions/answers for each passage
system_prompt = """You are creating a benchmark to evaluate a RAG (Retrieval-Augmented Generation) system.
Given a passage, generate two question/answer pairs, following these rules:
* Each question must contain enough explicit context to be understood and correctly 
  matched WITHOUT ever seeing the passage: name the relevant entities, subject, organization, 
  event, date, or topic directly in the question itself, exactly as a standalone user query would;
* NEVER refer to the source generically or deictically;
* Assume the RAG will see each question independently, in random order, with no memory of the other;
* Each question must be answerable using only the passage content (do not require outside knowledge);
* Each Gold Answer must be based strictly on the passage and approximately 600 characters long;
* The two questions for the same passage should be NOT directly and/or sequentially correlated;
* Do not modify the output format below, and do not add markdown symbols (e.g. "*") anywhere else.

OUTPUT FORMAT (exactly this, no extra text before or after):
Question 1: ...
Gold Answer 1: ...
"""
qa_pairs = []
for num, passage in enumerate(test_pages):
    page_text = passage["text"]
    prompt = f"Passage:\n{page_text}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    thinking_content, content = generation(tokenizer, model, messages, 16384, temperature=1)    # max tokens to be decided
    qa_pairs.append(content)

    print(f"Passage {str(num + 1)}:")
    print("content:", content, "\n-------------------------")

/mnt/eph/miniconda3/envs/nlp/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Passage 1: 
content: Question 1: What is the name of the naval academy located in Jablah?
Gold Answer 1: The name of the naval academy located in Jablah is JABLAH NAVAL ACADEMY. It is mentioned in the passage as part of the list of locations and facilities in Syria.

Question 2: At what time was one probable IL-76 observed active according to the passage?
Gold Answer 2: According to the passage, at 0006Z, one probable IL-76 was observed active IVO 36SYE681.4a221.4a. 
-------------------------
Passage 2: 
content: Question 1: What is the name of the invention developed by Miguel Angel Garcia Macias to avoid train collisions?
Gold Answer 1: Miguel Angel Garcia Macias invented the "FERRO TACTOMETRO y el FERROGEMACTOMETRO" to prevent train collisions at railway crossings. He later expanded this invention to prevent car accidents on roads but couldn't implement it due to lack of funding and failed to register the invention.

Question 2: What is the purpose of the "EQUINOCIS" mentioned in th

In [37]:
one_pair_questions = re.findall(r'Question 1:\s*(.+?)(?=\n|$)', '\n'.join(qa_pairs))
save_file("/mnt/eph/text_data/one_pair_questions.txt", one_pair_questions)

File /mnt/eph/text_data/one_pair_questions.txt saved


In [39]:
questions1 = re.findall(r'Question 1:\s*(.+?)(?=\n|$)', '\n'.join(qa_pairs))
gold_answers1 = re.findall(r'Gold Answer 1:\s*(.+)', '\n'.join(qa_pairs))
questions2 = re.findall(r'Question 2:\s*(.+?)(?=\n|$)', '\n'.join(qa_pairs))
gold_answers2 = re.findall(r'Gold Answer 2:\s*(.+)', '\n'.join(qa_pairs))

In [46]:
save_file("/mnt/eph/text_data/questions1.txt", questions1)

File /mnt/eph/text_data/questions1.txt saved


In [47]:
save_file("/mnt/eph/text_data/questions2.txt", questions2)

File /mnt/eph/text_data/questions2.txt saved


In [48]:
save_file("/mnt/eph/text_data/gold_answers1.txt", gold_answers1)

File /mnt/eph/text_data/gold_answers1.txt saved


In [49]:
save_file("/mnt/eph/text_data/gold_answers2.txt", gold_answers2)

File /mnt/eph/text_data/gold_answers2.txt saved


Here we use our RAG to generate answer to the given test-questions.

In [13]:
questions1 = load_file("/mnt/eph/text_data/questions1.txt")
questions2 = load_file("/mnt/eph/text_data/questions2.txt")

In [14]:
new_queries = questions1 + questions2

In [15]:
# Each query must come with a one-sentence instruction that describes the task
task = 'Given a document search query, retrieve relevant passages that answer the query'

queries = [get_detailed_instruct(task, query) for query in new_queries]

In [16]:
harrier_embedder = "microsoft/harrier-oss-v1-0.6b"
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

tokenizer = AutoTokenizer.from_pretrained(harrier_embedder, padding_side='left', cache_dir='tokenizers_cache')
model = AutoModel.from_pretrained(harrier_embedder, quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir='models_cache')
queriesEmbed = get_embeddings(queries)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

(**TO DO**: At the end we need to clean the following cells - stiamo indecizzando di nuovo tutti i documenti)

In [18]:
# retrieving the top-k documents for each query
numpydocs = docsEmbed.to(torch.float32).cpu().numpy()
numpyqueries = queriesEmbed.to(torch.float32).cpu().numpy()
index = faiss.index_factory(docsEmbed.shape[1], "Flat", faiss.METRIC_INNER_PRODUCT)   # cosine similarity
faiss.normalize_L2(numpydocs)
index.add(numpydocs)
faiss.normalize_L2(numpyqueries)

k=5
k_best_distance, k_best_index = index.search(numpyqueries, k)

**~TO DO~ DONE**: dare in input questions+retrieved document dei passaggi - modificare 'prompt'(vedi codice prompt) + modificare 'system_prompt' utilizzando quello finale della parte di generation (impostando una lunghezza massima nelle conclusioni)

**TO DO** (if we have time and sbates): change the names of model/tokenizer that atm are all the same.

In [ ]:
# qwen
qwen_generator = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(qwen_generator, padding_side='left', cache_dir='tokenizers_cache')
model = AutoModelForCausalLM.from_pretrained(qwen_generator, quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir="models_cache")

In [31]:
system_prompt = "We are impartial detectives investigating the latest FBI UAP documents.\
First tell what you know about the Query, without using the information in the provided documents.\
Then use the fresh information of the provided documents to accurately answer the Query. Give your answer using the following sections:\
# What we know;\
# Document information (address each document separately in this section);\
# Conclusions (use 600 characters for this section)"

answers = []
for i, query in enumerate(new_queries):
    retrDocs = [embedPages[int(idx)]["chunk_text"] for idx in k_best_index[i]]
    prompt = "Provided documents:\n"
    for j, doc in enumerate(retrDocs): prompt += f"Document {str(j + 1)}:\n{doc}.\n\n"
    prompt += f"\nQuery: {query}"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    content = generation(tokenizer, model, messages, 16384, temperature=1)[1]
    answers.append(content)

    print(f"Query {str(i + 1)}: {query}")
    print("content:", content, "\n-------------------------")

/mnt/eph/miniconda3/envs/nlp/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Query 1: What is the name of the naval academy located in Jablah?
content: What we know:
The query asks for the name of the naval academy located in Jablah.

Document information:
Document 1 contains a list of various locations and facilities, including "JABLAH NAVAL ACADEMY (36SYE651.4a191.4a)".

Document 2 is identical to Document 1, so it also contains the same information.

Document 3 mentions "JABLAH NAVAL ACADEMY (36SYE651.4a191.4a)" as part of a list of locations being monitored.

Document 4 is identical to Document 3, so it also contains the same information.

Document 5 does not mention Jablah or a naval academy there.

Conclusions:
The naval academy located in Jablah is the Jablah Naval Academy. Its coordinates are (36SYE651.4a191.4a). The name of the academy is confirmed in multiple documents, indicating its significance as a military training facility. 
-------------------------
Query 2: What is the name of the invention developed by Miguel Angel Garcia Macias to avoid trai

In [ ]:
import pickle  ## Siccome ho fatto di fretta ho usato pickle per salvare, possiamo eliminare per consegna o sostituire con save_text I guess
# with open("/mnt/eph/text_data/answersGioCheckpoint.txt", "wb") as fout: pickle.dump(answers, fout)

with open("/mnt/eph/text_data/answersGioCheckpoint.txt", "rb") as fin: answers = pickle.load(fin)

**TO DO - CONFIRM**: here we keep only the conclusions as rag_answers + we distinguish answer1/2 by the index (0-14 are 1, 15-30 are 2)

In [17]:
# to check 
# rag_answers = re.findall(r'Conclusions:\s*(.+?)(?=\n-+\n|$)', '\n'.join(answers))  # Sofia
rag_answers = re.findall(r'Conclusions[^:;]*[:;]\s*(.*)', '\n'.join(answers))  # Gio, controllato, funziona

In [18]:
gold_answers = load_file("/mnt/eph/text_data/gold_answers1.txt") + load_file("/mnt/eph/text_data/gold_answers2.txt")

Now that we have the questions, gold answers and answers we can use them as input for our test_generator that will act as a judge providing a score.

**TO DO**: check this prompt, i changed the score in a "semantically similar" score.

In [19]:
test_generator = "Qwen/Qwen2.5-Coder-14B-Instruct"
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

test_tokenizer = AutoTokenizer.from_pretrained(test_generator, padding_side='left', cache_dir='tokenizers_cache')
test_model = AutoModelForCausalLM.from_pretrained(test_generator, quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir="models_cache")

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

In [21]:
system_prompt = """You are a judge. Given a Gold answer (ground truth) and a Submitted answer, you will give me a semantic similarity score between the two.
The score ranges from 1 to 5 with higher scores indicating better alignment between the Submitted answer and the Gold answer. Give the reasons for the achieved score.
Output format:

Submitted answer score: .../5
Reasons: ...
"""
scores = []
for num in range(len(new_queries)):
    
    goldansw = gold_answers[num]
    ragansw = rag_answers[num]

    prompt = f"Gold answer: {goldansw}\nSubmitted answer: {ragansw}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    thinking_content, content = generation(test_tokenizer, test_model, messages, 16384, temperature=1)

    scores.append(content)

    print(f"Query {str(num + 1)}: {new_queries[num]}")
    print(f"Gold Answer: {goldansw}")
    print(f"RAG Answer: {ragansw}")
    print("content:", content, "\n-------------------------")

/mnt/eph/miniconda3/envs/nlp/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Query 1: What is the name of the naval academy located in Jablah?
Gold Answer: The name of the naval academy located in Jablah is JABLAH NAVAL ACADEMY. It is mentioned in the passage as part of the list of locations and facilities in Syria.
RAG Answer: The naval academy located in Jablah is the Jablah Naval Academy. Its coordinates are (36SYE651.4a191.4a). The name of the academy is confirmed in multiple documents, indicating its significance as a military training facility.
content: Submitted answer score: 4/5

Reasons:
- The Submitted answer correctly identifies the name of the naval academy as "Jablah Naval Academy," which aligns with the Gold answer.
- It provides additional relevant information, such as the coordinates of the academy, which enhances the answer's detail.
- The mention that the name is confirmed in multiple documents adds credibility and supports the academy's status as a military training facility, which is consistent with the context provided in the Gold answer.
-

In [24]:
import pickle  ## Siccome ho fatto di fretta ho usato pickle per salvare, possiamo eliminare per consegna o sostituire con save_text I guess
# with open("/mnt/eph/text_data/scoresJudge.txt", "wb") as fout: pickle.dump(scores, fout)

with open("/mnt/eph/text_data/scoresJudge.txt", "rb") as fin: scores = pickle.load(fin)

In [ ]:
for passage in scores:
    scores1 = re.findall(r'Question (\d+):.*?RAG answer score: (\d+)/100', scores, re.DOTALL)

In [ ]:
scores1

# Acknowledgements

- Dataset: https://huggingface.co/datasets/MTSlive/war-gov-uap-release-1.
- Qwen3 web reference: https://qwen.ai/blog?id=qwen3-embedding.
- Qwen3 Article: https://arxiv.org/pdf/2506.05176.

# Appendix

## Appendix A - Retrieval

We can infer from the retrieved documents below that overall both embedders seem to work properly. For some of the queries they may even retrieve the same passages (with different scorings).

However, generic tasks such as: "What was spotted in the sky for the first time?" might not be solvable problems on such a large dataset if there is not a clear answer in the text. A specific pipeline or instruction prompt might help in processing these tasks.

Since Harrier is a far lighter model, and in our opinion retrieved and classified better the passages for some of the tasks, we decided to use this model as the embedder of our RAG.

In [25]:
## Harrier
for i, query in enumerate(raw_queries):
    print(query)
    for j, idx in enumerate(k_best_index[i]):
        print(f"\nscore: {k_best_distance[i,j]}; idx: {idx}\n{embedPages[idx]["chunk_text"]}")
    print("\n\n\n")

What are Jesus's powers?

score: 0.5771710872650146; idx: 3266
The life of Jesus, now too, becomes clearer when these things are re-membered. His powers of levitation, His ability to pass through doors, walk on water, and heal the sick, are the essential attributes of men from outer space. They will also be ours someday when we will be "free like birds" (Ezk. 13:20). At Jesus' birth the celestial army came quite close to earth. A space-man appeared to the shepherds and the "glory" of the Lord, with the usual signs of His presence, shone around them. There was a multitude of the heavenly army with this space-man. And after their cosmic announcement, the music of their space-ships was heard as they again disappeared into space.

Jesus' ascension is described as "a cloud (or space-ship) received Him out of their sight" (Acts 1:9). His coming again is to be in the same manner. "Then will appear the sign of the Son of man in heaven (space) coming on the clouds (space-ships) of heaven (space

In [24]:
## Qwen
for i, query in enumerate(raw_queries):
    print(query)
    for j, idx in enumerate(k_best_index[i]):
        print(f"\nscore: {k_best_distance[i,j]}; idx: {idx}\n{embedPages[idx]["chunk_text"]}")
    print("\n\n\n")

What are Jesus's powers?

score: 0.623723030090332; idx: 3272
The life of Jesus, now too, becomes clearer when these things are re-membered. His powers of levitation, His ability to pass through doors, walk on water, and heal the sick, are the essential attributes of men from outer space. They will also be ours someday when we will be "free like birds" (Ezk. 13:20). At Jesus' birth the celestial army came quite close to earth. A space-man appeared to the shepherds and the "glory" of the Lord, with the usual signs of His presence, shone around them. There was a multitude of the heavenly army with this space-man. And after their cosmic announcement, the music of their space-ships was heard as they again disappeared into space.

Jesus' ascension is described as "a cloud (or space-ship) received Him out of their sight" (Acts 1:9). His coming again is to be in the same manner. "Then will appear the sign of the Son of man in heaven (space) coming on the clouds (space-ships) of heaven (space)

## Appendix B - Generation

In [53]:
system_prompt = "We are impartial detectives investigating the latest FBI UAP documents.\
First tell what you know about the Query, without using the information in the provided documents.\
Then use the fresh information of the provided documents to accurately answer the Query. Give your answer using the following sections:\
# What we know;\
# Document information (address each document separately);\
# Conclusions"

for i, query in enumerate(raw_queries):
    retrDocs = [embedPages[int(idx)]["chunk_text"] for idx in k_best_index[i]]
    prompt = "Provided documents:\n"
    for j, doc in enumerate(retrDocs): prompt += f"Document {str(j + 1)}:\n{doc}.\n\n"
    prompt += f"\nQuery: {query}"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    print(f"Query: {query}\n")
    print("Answer", generation(tokenizer, model, messages, 16384, temperature=1)[1], "\n\n")

Query: What are Jesus's powers?

Answer What we know:
Jesus is described in various religious texts as having supernatural abilities, including the power to perform miracles, heal the sick, walk on water, pass through doors, and levitate. These powers are interpreted by some as being similar to the abilities of extraterrestrial beings, and are seen as essential attributes of a divine figure. Additionally, there are references to Jesus's ascension and his return being described in terms of space travel, with imagery of space ships, clouds, and other cosmic elements.

Document information (address each document separately):
Document 1 describes Jesus's powers as including levitation, passing through doors, walking on water, and healing the sick. It also references the idea that these powers are similar to those of extraterrestrial beings, and that Jesus's ascension and return are described in terms of space travel, with imagery of space ships, clouds, and other cosmic elements.

Document

## Appendix C - Embedding code

Python script run to embed the documents.

---

```python
#!/usr/bin/env python

from datasets import load_dataset, get_dataset_split_names, get_dataset_config_names, load_dataset_builder, Dataset
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
import torch
from tqdm import tqdm
import numpy as np
from pathlib import Path
import re
import torch.nn.functional as F
from torch import Tensor
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pickle


# function to extract the last token (EOS) that is a compressed representation of the whole input (instruction+query)
def last_token_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]


def main():
    modelName = "microsoft/harrier-oss-v1-0.6b"  # "Qwen/Qwen3-Embedding-4B"
    # batch size is chosen to be the largest possible to fit the GPU memory
    batch_size = 140  # 40
    outFile = "harrier06DocsEmbed.pt"  # "qwen4DocsEmbed.pt"

    ufo_dataset = "MTSLIVE/war-gov-uap-release-1"
    pages = load_dataset(ufo_dataset, "pages", split="train")

    IDS = set(pages["document_id"])

    # Discard single paged documents containing no information
    pattern = re.compile("fbi-photo|nasa-uap-vm|fbi-september-2023-sighting-composite-sketch")  # compile pattern to match discarded documents
    useIDS = set(ID for ID in IDS if not pattern.match(ID))  # discard matches

    filterPages = pages.filter(lambda x: x["document_id"] in useIDS and x["text"] != '')

    # defining the embedding model and the relative tokenizer
    quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
    tokenizer = AutoTokenizer.from_pretrained(modelName, padding_side='left', cache_dir='tokenizers_cache')
    model = AutoModel.from_pretrained(modelName, quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir="models_cache")

    # Check if we are running on the GPU
    if model.device.type != "cuda": raise RuntimeError("NOT CUDA????")
    print("model loaded")

    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer, chunk_size=512, chunk_overlap=77)

    pages_chunked = []

    for page in filterPages:
        page_text = page['text']
        chunk_list = text_splitter.split_text(page_text)

        for i, chunk in enumerate(chunk_list):
            pages_chunked.append({
                'document_id': page['document_id'],
                'page_no': page['page_no'],
                'chunk_text': chunk
        })


    embeddings = []
    for i in tqdm(range(0, len(pages_chunked), batch_size)):
        usePages = pages_chunked[i:i+batch_size]
        input_texts = [page["chunk_text"] for page in usePages]

        max_length = 512

        # Tokenize the input texts
        batch_dict = tokenizer(
            input_texts,
            padding="longest",
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad(): outputs = model(**batch_dict)
        embeddings.append(last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask']).to("cpu"))

        # Make absolutely certain that GPU memory is cleared for the next iteration
        del batch_dict, outputs
        torch.cuda.empty_cache()

    # Save the embeddings
    torch.save(torch.cat(embeddings), outFile, pickle_protocol=5)

if __name__ == "__main__": main()
```

## Appendix D - Evaluation Prompting

In [14]:
# two quastions/answers for each passage
system_prompt = """You are creating a benchmark for evaluating a RAG system.
Given a passage, generate exactly two question/answer pair.
Output format:

Question 1: ...
Gold Answer 1 (600 characters): ...
Question 2: ...
Gold Anser 2 (600 characters): ...
"""
qa_pairs = []
for num, passage in enumerate(test_pages):
    page_text = passage["text"]
    prompt = f"Passage:\n{page_text}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    thinking_content, content = generation(test_tokenizer, test_model, messages, 16384)    # max tokens to be decided
    qa_pairs.append(content)

    print(f"Passage {str(num + 1)}:", thinking_content)
    print("content:", content, "\n-------------------------")

/mnt/eph/miniconda3/envs/nlp/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Passage 1: 
content: Question 1: What was the purpose of the Office Memorandum dated July 16, 1947?
Gold Answer 1 (600 characters): The purpose of the Office Memorandum dated July 16, 1947, was to request an investigation of Richard F. Shaver, who was believed to have information about the origin of "flying saucers." This conclusion was drawn based on an unsigned telegram received by HQ. AAF on July 9, 1947, indicating that Shaver might have relevant information, and reports of flying saucers observed near Lily Lake, Illinois, around the same time.

Question 2: What specific documents were attached to the Office Memorandum?
Gold Answer 2 (600 characters): The Office Memorandum attached three documents: 
a. An unsigned telegram received by HQ. AAF on July 9, 1947, mentioning Richard F. Shaver and his possible connection to the origin of "flying saucers."
b. A report detailing observations of flying saucers by four witnesses while in two airplanes over southern Wisconsin.
c. A map showin

In [ ]:
# two quastions/answers for each passage
system_prompt = """You are creating a benchmark to evaluate a RAG system.
Given a passage, generate exactly two question/answer pair taking into account the following constraints:
* The questions should be based on the specific content of the passage;
* The questions will be prompted to the RAG system, which has access to a much larger corpus of documents, and has not retrieved the given passage yet;
* Each answer must be 600 characters long;
Output format:

Question 1: ...
Gold Answer 1: ...
Question 2: ...
Gold Answer 2: ..."""
qa_pairs = []
for num, passage in enumerate(test_pages):
    page_text = passage["text"]
    prompt = f"Passage:\n{page_text}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    thinking_content, content = generation(test_tokenizer, test_model, messages, 16384, temperature=1)    # max tokens to be decided
    qa_pairs.append(content)

    print(f"Passage {str(num + 1)}:", thinking_content)
    print("content:", content, "\n-------------------------")

/mnt/eph/miniconda3/envs/nlp/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Passage 1: 
content: **Question 1:**  
What document was attached to the memorandum dated 16 July 47, and what did it contain?

**Gold Answer 1:**  
The memorandum dated 16 July 47 contained three attachments:

a. An unsigned telegram received by HQ. AAF on 9 July 1947. This telegram mentioned that Richard F. Shaver, located in Lilly Lake, Illinois, may have information concerning the origin of "flying saucers."

b. A report detailing observations of flying saucers by four witnesses who spotted these objects in two airplanes while flying over southern Wisconsin.

c. A map indicating the proximity of the locations where the flying saucers were reported to have been seen in relation to Lilly Lake, Illinois.

These attachments provided evidence suggesting a connection between Richard F. Shaver and reports of flying saucers, prompting further investigation into whether Shaver possessed any information about the origin of these unidentified aerial phenomena.

**Question 2:**  
Why was Richa

To correct the recurrent issue, the following `system_prompt` has been given to Claude Sonnet 5 with the following prompt:<br>
```
help me enhance the prompt of a model.
The model has to create two pairs of question/answer based for each input document. Then the same question (without the answer) will be given to my RAG that has to retrieve the document with cos-similarity and then answer. The model receive the document on which has to create the questions, instead my RAG won't receive the same document (but only the one it retrieves from the given question with similarity search). 
The problem is that the model is creating questions like " ... in the given text?", "... in the document?", so refering to the given document but then the RAG won't know which document is refering to or might be that it has retrieved a different document.
At the moment my prompt for the model is the following:
system_prompt = """..."""
```

In [20]:
# two quastions/answers for each passage
system_prompt = """You are creating a benchmark to evaluate a RAG system.
Given a passage, generate exactly two question/answer pair taking into account the following constraints:
* The questions should be based on the specific content of the passage;
* The questions will be prompted to the RAG system, which has access to a much larger corpus of documents, and has not retrieved the given passage yet;
* The questions must avoid implicit reference to the passage, if needed make only explicit reference;
* Assume the RAG will not see the two questions together (the questions will be given in random order);
* Do NOT make questions like: "... in this document?"
* Each answer must be 600 characters long;

Use the following Output format:
Question 1: ...
Gold Answer 1: ...
Question 2: ...
Gold Answer 2: ..."""
qa_pairs = []
for num, passage in enumerate(test_pages):
    page_text = passage["text"]
    prompt = f"Passage:\n{page_text}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    thinking_content, content = generation(tokenizer, model, messages, 16384, temperature=1)    # max tokens to be decided
    qa_pairs.append(content)

    print(f"Passage {str(num + 1)}:", thinking_content)
    print("content:", content, "\n-------------------------")

Passage 1: 
content: Question 1: What are the coordinates and designations of the Syrian Navy Operations Areas mentioned in the given text?
Gold Answer 1: The Syrian Navy Operations Areas mentioned in the passage are designated as follows: SYRIAN NAVY OPS AREA (36SYD391.4a541.4a), SYRIAN NAVY OPS AREA (36SYE371.4a411.4a), SYRIAN NAVY OPS AREA (36SXE681.4a401.4a), and SYRIAN NAVY OPS AREA (36SXD701.4a531.4a). These areas are part of a broader operational region in the Middle East involving Israel, Lebanon, and Syria.

Question 2: Describe the military assets and activities observed near Latakia Naval Mon DSA (36SYE211.4a751.4a) according to the information provided.
Gold Answer 2: Near the Latakia Naval Mon DSA (36SYE211.4a751.4a), various military assets were observed. These include a probable Gorshkov FFG heading east at IVO 36STD291.4a771.4a, a probable UdaloY I DD, hull 626, also heading east at IVO 36STC861.4a941.4a, and a probable Slava CG, hull 055, moving east at IVO 36STD791.4a

KeyboardInterrupt: 

But still the problem does not seem solved (see below):

In [21]:
# two quastions/answers for each passage
system_prompt = """You are creating a benchmark to evaluate a RAG (Retrieval-Augmented Generation) system.

Given a passage, generate exactly two question/answer pairs, following these rules:

CONTEXT INDEPENDENCE (most important rule):
* The RAG system will receive ONLY the question, in isolation, with no access to this passage. 
  It must first retrieve a document via similarity search based on the question text alone, 
  then answer using the retrieved content.
* This means each question must contain enough explicit context to be understood and correctly 
  matched WITHOUT ever seeing the passage: name the relevant entities, subject, organization, 
  event, date, or topic directly in the question itself, exactly as a standalone user query would.
* NEVER refer to the source generically or deictically. Forbidden patterns include (non-exhaustive): 
  "this/the document", "this/the text", "this/the passage", "this/the letter", "this/the article", "the given text", 
  "according to the passage", "mentioned above", "in this excerpt", "the author states that...".
* Test yourself: if you deleted the passage, could a person unfamiliar with it still know what 
  the question is about and roughly what to search for? If not, rewrite it.

EXAMPLES:
* Bad:  "What was the main reason given in the letter for the delay?"
* Good: "What reason did [Company X] give for the delay in the Q3 shipment to [Client Y]?"
* Bad:  "According to the document, when was the policy enacted?"
* Good: "When was the [specific policy name] enacted in [country/context]?"
(Replace bracketed placeholders with the actual specific entities from the passage — do not 
leave them generic.)

ADDITIONAL CONSTRAINTS:
* The two questions must be based on specific, distinct content from the passage (not overlapping facts).
* Assume the RAG will see each question independently, in random order, with no memory of the other.
* Each question must be answerable using only the passage content (do not require outside knowledge).
* Each Gold Answer must be based strictly on the passage and approximately 600 characters long.
* Do not modify the output format below, and do not add markdown symbols (e.g. "*") anywhere else.

OUTPUT FORMAT (exactly this, no extra text before or after):
Question 1: ...
Gold Answer 1: ...
Question 2: ...
Gold Answer 2: ...
"""
qa_pairs = []
for num, passage in enumerate(test_pages):
    page_text = passage["text"]
    prompt = f"Passage:\n{page_text}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    thinking_content, content = generation(tokenizer, model, messages, 16384, temperature=1)    # max tokens to be decided
    qa_pairs.append(content)

    print(f"Passage {str(num + 1)}:", thinking_content)
    print("content:", content, "\n-------------------------")

Passage 1: 
content: Question 1: What is the location of the Syrian Navy Ops Area mentioned in the passage?
Gold Answer 1: The Syrian Navy Ops Area is located at coordinates 36SYD391.4a541.4a.

Question 2: When was the A-50U Mainstay and two IL-38 aircraft observed parked according to the passage?
Gold Answer 2: According to the passage, at 2120Z, one A-50U Mainstayer and two IL-38 aircraft were observed parked at coordinates IVO 36SYE681.4a221.4a. 
-------------------------


KeyboardInterrupt: 

At this point, the prompt seems too much focused on the context independence problem that, paradoxically, could be the reason why the model is still performing badly. We try to simplify the prompt:
- asking for only one question/answer pair
- removing the exmaples
- not citing the forbidden patterns (that could be the main issue)

In [26]:
# two quastions/answers for each passage
system_prompt = """You are creating a benchmark to evaluate a RAG (Retrieval-Augmented Generation) system.
Given a passage, generate one question/answer pair, following these rules:
* Each question must contain enough explicit context to be understood and correctly 
  matched WITHOUT ever seeing the passage: name the relevant entities, subject, organization, 
  event, date, or topic directly in the question itself, exactly as a standalone user query would;
* NEVER refer to the source generically or deictically;
* Assume the RAG will see each question independently, in random order, with no memory of the other;
* Each question must be answerable using only the passage content (do not require outside knowledge);
* Each Gold Answer must be based strictly on the passage and approximately 600 characters long;
* Do not modify the output format below, and do not add markdown symbols (e.g. "*") anywhere else.

OUTPUT FORMAT (exactly this, no extra text before or after):
Question 1: ...
Gold Answer 1: ...
"""
qa_pairs = []
for num, passage in enumerate(test_pages):
    page_text = passage["text"]
    prompt = f"Passage:\n{page_text}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    thinking_content, content = generation(tokenizer, model, messages, 16384, temperature=1)    # max tokens to be decided
    qa_pairs.append(content)

    print(f"Passage {str(num + 1)}:", thinking_content)
    print("content:", content, "\n-------------------------")

Passage 1: 
content: Question 1: What was the observed aircraft at IVO 36SYE681.4a221.4a between 0006Z and 0029Z?
Gold Answer 1: At 0006Z, one probable IL-76 was observed active at IVO 36SYE681.4a221.4a. At 0011Z, one probable SU-27/35 was observed landing at the same location. At 0029Z, one probable SU-27/35 was observed taking off to the south. 
-------------------------
Passage 2: 
content: Question 1: ¿Qué invento ideográfico amplió el Ferrotactómetro y el Ferrogemactómetro para evitar choques de autos en carreteras?
Gold Answer 1: El invento ideográfico mencionado es el Equinocis, que fue ampliado para evitar choques de autos en carreteras. El Equinocis es un dispositivo que permite precisar la distribución y repartición del tiempo en los relojes de todo el mundo. Este invento se encuentra en la Biblioteca del Pueblo de un puerto específico. 
-------------------------
Passage 3: 
content: Question 1: What is the launch facility for the GLORY TRIP 65B mission on 27 May 70?
Gold Ans